In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from src.datasets.multi_season_dataset import build_multi_season_dataset


raw_X, raw_y = build_multi_season_dataset(
    seasons=list(range(2003, 2026))
)

print(f"\nX shape: {raw_X.shape}")
print(f"Number of features: {len(raw_X.columns)}")
print("Train features:")
print(list(raw_X.columns))

Starting season 2003...
Finished season 2003: 4413 rows built, 203 skipped, 4616 raw games
Starting season 2004...
Finished season 2004: 4364 rows built, 207 skipped, 4571 raw games
Starting season 2005...
Finished season 2005: 4465 rows built, 210 skipped, 4675 raw games
Starting season 2006...
Finished season 2006: 4541 rows built, 216 skipped, 4757 raw games
Starting season 2007...
Finished season 2007: 4835 rows built, 208 skipped, 5043 raw games
Starting season 2008...
Finished season 2008: 4954 rows built, 209 skipped, 5163 raw games
Starting season 2009...
Finished season 2009: 5038 rows built, 211 skipped, 5249 raw games
Starting season 2010...
Finished season 2010: 5051 rows built, 212 skipped, 5263 raw games
Starting season 2011...
Finished season 2011: 5033 rows built, 213 skipped, 5246 raw games
Starting season 2012...
Finished season 2012: 5039 rows built, 214 skipped, 5253 raw games
Starting season 2013...
Finished season 2013: 5112 rows built, 208 skipped, 5320 raw games

In [4]:
from src.feature_engineering.feature_analysis import FeatureAnalyzer
analyzer = FeatureAnalyzer(raw_X)
print("Ready to analyze the features candidates and recommend data engineering steps.")

Ready to analyze the features candidates and recommend data engineering steps.


In [5]:
# Identify pairs of highly correlated variables
corr = analyzer.analyze_collinearity(threshold=0.90)

Highly Correlated Feature Pairs
0.992 team_1_last_10_avg_opponent_win_pct <-> team_1_avg_opponent_win_pct
0.992 team_2_last_10_avg_opponent_win_pct <-> team_2_avg_opponent_win_pct
0.991 team_1_last_10_avg_opponent_point_diff_pg <-> team_1_avg_opponent_point_diff_pg
0.990 team_2_last_10_avg_opponent_point_diff_pg <-> team_2_avg_opponent_point_diff_pg
0.979 team_1_games_played <-> team_2_games_played
0.968 team_2_last_5_avg_opponent_win_pct <-> team_2_last_10_avg_opponent_win_pct
0.968 team_1_last_5_avg_opponent_win_pct <-> team_1_last_10_avg_opponent_win_pct
0.966 DayNum <-> team_2_games_played
0.965 DayNum <-> team_1_games_played
0.964 team_1_last_5_avg_opponent_point_diff_pg <-> team_1_last_10_avg_opponent_point_diff_pg
0.964 team_2_last_5_avg_opponent_point_diff_pg <-> team_2_last_10_avg_opponent_point_diff_pg
0.962 team_2_off_rebounds_pg <-> team_2_last_10_off_rebounds_pg
0.960 team_1_off_rebounds_pg <-> team_1_last_10_off_rebounds_pg
0.960 team_2_last_5_avg_opponent_win_pct <-> tea

In [ ]:
# Fix And Remove Redundancies

In [6]:
# Find top performing pairwise interaction components
interactions = analyzer.analyze_interactions(raw_y, top_features=50)

KeyboardInterrupt: 

In [ ]:
# Now we can combine them into new, better features

In [ ]:
# Get a summary and update the results
analyzer.summary()
raw_X = analyzer.get_data()

In [ ]:
from src.preprocessing.preprocess import preprocess_train_test
from src.preprocessing.splits import split_raw_by_season

# Perform a test-train split
raw_X_train, raw_X_test, raw_y_train, raw_y_test = split_raw_by_season(
    raw_X,
    raw_y,
    test_seasons=[2025],
)

# And perform pre-processing of the data into symmetric, differential columns
X_train, X_test, y_train, y_test = preprocess_train_test(
    raw_X_train,
    raw_X_test,
    raw_y_train,
    raw_y_test,
    prefix1="team_1_",
    prefix2="team_2_",
    invert_cols=["team_1_location"],
    diff_suffix="_diff",
    drop_base_features=True
)

In [ ]:
from src.feature_engineering.feature_selector import FeatureSelector

selector = FeatureSelector(X_train, y_train)
print("Ready to begin final features selection.")

In [ ]:
import warnings

# Turn of warnings and start with Lasso
warnings.filterwarnings("ignore", category=FutureWarning, module="sklearn.linear_model")
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.linear_model")
selector.run_lasso(target_features=35, print_results=True, plot_results=True)

In [ ]:
# Continue to RFE backwards elimination
selector.run_rfe(target_features=35, print_results=True, plot_results=True)

In [ ]:
# Finish with tree importance for non-linear patterns
selector.run_tree_importance(target_features=35, print_results=True, plot_results=True)

In [ ]:
# Save only the features that two methods pointed out as relevant
selector.finalize_selection(vote_threshold=2, print_results=True)

# And update the data frames accordingly
X_train = selector.transform(X_train)
X_test = selector.transform(X_test)

In [ ]:
from pathlib import Path

# Project root is two levels above src/notebook
data_dir = Path("../../data")
data_dir.mkdir(parents=True, exist_ok=True)

X_train.to_csv(data_dir / "X_train.csv", index=False)
X_test.to_csv(data_dir / "X_test.csv", index=False)
y_train.to_csv(data_dir / "y_train.csv", index=False)
y_test.to_csv(data_dir / "y_test.csv", index=False)

print(f"Saved dataframes to {data_dir.resolve()}")